[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/06_How_Gradient_Descent_Learns.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/06_How_Gradient_Descent_Learns.ipynb)

# How Gradient Descent Learns — Error, Gradient, and Weight Updates Step by Step

Every weight in a model starts as a random guess. Gradient descent is the process that corrects those guesses, one step at a time, using the errors the model makes.

This notebook walks through that process from scratch — no sklearn training loop, just NumPy — so every calculation is visible.

---

**Four steps this notebook makes concrete**

1. **Start** with random weights → make predictions → measure errors
2. **Gradient**: each observation casts a vote — how much should each weight change?
3. **Update**: nudge every weight a small step in the direction that reduces error
4. **Repeat** until the loss stops falling — weights have converged to their true values

The connecting thread: the same `weight × feature_value` arithmetic that drives predictions also drives learning.
A feature with value 0 is silent in the prediction **and** casts a zero vote in the gradient.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)
print('Ready.')


In [ ]:
n = 400

age            = np.random.uniform(25, 65, n)
bmi            = np.random.uniform(18, 40, n)
activity_level = np.random.choice([1, 2, 3, 4], n, p=[0.25, 0.35, 0.25, 0.15]).astype(float)
smoker         = np.random.choice([0, 1], n, p=[0.75, 0.25]).astype(float)
has_chronic    = np.random.choice([0, 1], n, p=[0.70, 0.30]).astype(float)

TRUE_INTERCEPT = 3_000.0
TRUE_W = np.array([50.0, 30.0, -200.0, 3_500.0, 2_500.0])
FEATURES = ['age', 'bmi', 'activity_level', 'smoker', 'has_chronic']
FTYPE    = ['Continuous', 'Continuous', 'Ordinal', 'Binary', 'Binary']

X_raw = np.column_stack([age, bmi, activity_level, smoker, has_chronic])
y     = (TRUE_INTERCEPT
         + X_raw @ TRUE_W
         + np.random.normal(0, 300, n))

df = pd.DataFrame(X_raw, columns=FEATURES)
df['annual_premium'] = y

print(f'Dataset: {n} policyholders, {len(FEATURES)} features')
print(f'Premium range: ${y.min():,.0f}  –  ${y.max():,.0f}')
print()
print(f"  {'Feature':<18} {'Type':<12} {'True weight':>12}")
print("  " + "-" * 46)
for f, ft, tw in zip(FEATURES, FTYPE, TRUE_W):
    print(f"  {f:<18} {ft:<12} {tw:>+12,.0f}")
print(f"  {'intercept':<18} {'':12} {TRUE_INTERCEPT:>+12,.0f}")


## Part 1 — Start: Random Weights, Large Errors

Before training, all weights are zero (or small random values).
The model has no knowledge of the data — its predictions will be far off.

The **error** for one observation is simply:

```
error = actual − predicted
```

Positive error → model under-predicted → weights need to go **up**.  
Negative error → model over-predicted → weights need to go **down**.

The **loss** summarises how wrong the model is across all observations:

```
MSE = (1/n) × Σ( errorᵢ² )
```

MSE is always positive and zero only when every prediction is perfect.
The goal of training is to find the weights that make MSE as small as possible.


In [ ]:
# Weights start at zero — model knows nothing
w_init = np.zeros(len(FEATURES))
b_init = 0.0

y_pred_init = X_raw @ w_init + b_init
errors_init = y - y_pred_init
mse_init    = np.mean(errors_init ** 2)

print("Initial weights (all zero):")
for f, w in zip(FEATURES, w_init):
    print(f"  w_{f:<18} = {w:.1f}")
print()
print(f"{'Obs':>4}  {'Actual ($)':>12}  {'Predicted ($)':>14}  {'Error ($)':>12}")
print("-" * 46)
for i in range(6):
    print(f"  {i:>2}  {y[i]:>12,.0f}  {y_pred_init[i]:>14,.0f}  {errors_init[i]:>12,.0f}")
print()
print(f"Initial MSE: {mse_init:,.0f}   RMSE: ${np.sqrt(mse_init):,.0f}")
print("Every prediction is $0 — error equals the full premium for every person.")

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(errors_init, bins=40, color='#e57373', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=1.5)
ax.set_xlabel('Error = actual − predicted  ($)')
ax.set_title('Error distribution at iteration 0 (all weights = 0)\nAll errors are positive and large — model under-predicts everything', fontsize=10)
plt.tight_layout()
plt.show()


## Part 2 — Gradient: Each Observation Votes on Each Weight

The gradient tells us which direction and by how much to move each weight.

For a single observation `i`, its **vote** on weight `j` is:

```
vote_ij  =  error_i  ×  feature_value_ij
```

---

### Where does this formula come from? (The derivative)

This is not arbitrary — it is the calculus derivative of the loss with respect to the weight.

**Step 1 — write down the prediction and squared error for one observation:**

```
ŷᵢ    = Σⱼ( wⱼ × xᵢⱼ ) + b        ← dot product of weights and features

lossᵢ = (ŷᵢ − yᵢ)²                 ← squared error for observation i
```

**Step 2 — ask: "how much does the loss change if we nudge weight wⱼ a tiny amount?"**

That is the derivative `d(lossᵢ) / d(wⱼ)`.  Applying the chain rule (outer × inner derivative):

```
d(lossᵢ) / d(wⱼ)  =  2 × (ŷᵢ − yᵢ) × xᵢⱼ
```

Since `errorᵢ = yᵢ − ŷᵢ`, we have `(ŷᵢ − yᵢ) = −errorᵢ`, so:

```
d(lossᵢ) / d(wⱼ)  =  −2 × errorᵢ × xᵢⱼ
```

**Step 3 — average over all n observations (MSE = mean of squared errors):**

```
d(MSE) / d(wⱼ)  =  −(2/n) × Σᵢ( errorᵢ × xᵢⱼ )
```

The `2` is absorbed into the learning rate in practice.  What remains is:

```
gradient_j  ∝  − average of ( errorᵢ × xᵢⱼ ) across all observations i
```

**`error × feature_value` is not a heuristic — it is literally the slope of the loss surface
with respect to that weight.**  Moving opposite this slope (`w -= lr × gradient`) descends
the loss toward the minimum.

---

### Intuition from the formula

- `error > 0` (under-predicted) and `feature_value > 0` → derivative is negative → subtract a negative → **weight goes up** ✓  
- `error > 0` (under-predicted) and `feature_value = 0` → derivative is **zero** → weight gets no update ← the silent feature rule  
- `error < 0` (over-predicted) → derivative flips sign → **weight goes down** ✓  
- Larger `feature_value` → steeper slope → bigger update step for that weight

**The silent feature rule**: if `feature_value_ij = 0`, the derivative for weight `j` from observation `i` is exactly zero.
That row contributes nothing to the gradient — it cannot move that weight regardless of the error.

The **gradient for weight j** averaged across all n observations:

```
gradient_j  =  −(2/n) × Σᵢ( errorᵢ × feature_value_ij )
```

In [ ]:
# Pick one observation — a non-smoker with chronic condition
mask = (df['smoker'] == 0) & (df['has_chronic'] == 1)
OBS  = df[mask].index[0]

actual    = y[OBS]
predicted = y_pred_init[OBS]
error     = actual - predicted
x_obs     = X_raw[OBS]

print(f"=== Observation {OBS} ===")
print(f"  smoker = {int(x_obs[3])}   has_chronic = {int(x_obs[4])}")
print(f"  Actual premium    = ${actual:,.0f}")
print(f"  Predicted premium = ${predicted:,.0f}")
print(f"  Error             = ${error:,.0f}  (positive → model under-predicted → weights should go UP)")
print()
print(f"  {'Feature':<18} {'Type':<12} {'Value':>8}  {'Vote = error × value':>24}  {'':>10}")
print("  " + "-" * 72)
for feat, ftype, val in zip(FEATURES, FTYPE, x_obs):
    vote   = error * val
    silent = "  ← SILENT (value=0, no vote)" if val == 0 else ""
    print(f"  {feat:<18} {ftype:<12} {val:>8.1f}  {vote:>24,.1f}{silent}")

# ── Gradient vote bar chart ────────────────────────────────────────────────────
votes = error * x_obs

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#cccccc' if v == 0 else ('#4CAF50' if v > 0 else '#e57373') for v in votes]
bars = ax.barh(FEATURES, votes, color=colors, edgecolor='white', height=0.55)
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Gradient vote  (error × feature value)')
ax.set_title(f'Observation {OBS}: gradient vote for each weight\nGray = SILENT (value=0)   Green = increase weight   Red = decrease', fontsize=10)

for bar, (feat, val, vote) in zip(bars, zip(FEATURES, x_obs, votes)):
    y_mid = bar.get_y() + bar.get_height() / 2
    if val == 0:
        ax.text(max(votes) * 0.02, y_mid,
                f'{error:,.0f} × {val:.0f} = 0   SILENT',
                va='center', ha='left', fontsize=8, color='#888', style='italic')
    else:
        xpos = vote + max(votes) * 0.01 if vote >= 0 else vote - max(votes) * 0.01
        ha   = 'left' if vote >= 0 else 'right'
        ax.text(xpos, y_mid, f'{error:,.0f} × {val:.1f} = {vote:,.0f}',
                va='center', ha=ha, fontsize=8)

ax.legend(handles=[
    Patch(facecolor='#4CAF50', label='Vote to increase weight'),
    Patch(facecolor='#e57373', label='Vote to decrease weight'),
    Patch(facecolor='#cccccc', label='Silent — feature value is 0'),
], fontsize=9)
plt.tight_layout()
plt.show()

print()
print(f"smoker=0 → this non-smoker casts ZERO vote on w_smoker.")
print(f"The model cannot learn about the smoker premium from this person.")
print(f"It needs smoker=1 observations to learn that w_smoker should be ~{TRUE_W[3]:,.0f}.")


## Part 3 — Averaging Votes Across All Observations

One observation's vote is noisy. The gradient for weight j across the full dataset is the **average vote**:

```
gradient_j  =  −(2/n) × Σᵢ( errorᵢ × xᵢⱼ )
```

Think of it as a poll:
- Each observation casts a vote proportional to `error × feature_value`
- Observations with large errors vote louder
- Observations where the feature is 0 abstain
- The gradient is the average opinion

For binary features like `smoker`:
- The 75% of rows where `smoker = 0` abstain completely
- Only the 25% of smokers vote on `w_smoker`
- Their votes are averaged with the 0s → the effective gradient is diluted but still correct


In [ ]:
# Show the distribution of votes for two features: age vs smoker
votes_age    = errors_init * X_raw[:, 0]    # error × age for every row
votes_smoker = errors_init * X_raw[:, 3]    # error × smoker for every row

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(votes_age, bins=40, color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].axvline(votes_age.mean(), color='#e57373', lw=2.5,
                label=f'mean vote = {votes_age.mean():,.0f}')
axes[0].set_title('Gradient votes for w_age\nEvery row votes (age is never 0)', fontsize=10)
axes[0].set_xlabel('error × age')
axes[0].legend(fontsize=9)

axes[1].hist(votes_smoker[X_raw[:, 3] == 0], bins=1,
             color='#cccccc', edgecolor='white', alpha=0.9, label='smoker=0 (abstain)')
axes[1].hist(votes_smoker[X_raw[:, 3] == 1], bins=30,
             color='#FF7043', edgecolor='white', alpha=0.85, label='smoker=1 (active vote)')
axes[1].axvline(votes_smoker.mean(), color='#e57373', lw=2.5,
                label=f'mean vote = {votes_smoker.mean():,.0f}')
axes[1].set_title('Gradient votes for w_smoker\n75% of rows abstain (smoker=0) — only smokers vote', fontsize=10)
axes[1].set_xlabel('error × smoker')
axes[1].legend(fontsize=9)

plt.suptitle('Each observation casts one vote per weight — abstentions count as zero',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# Show the full gradient vector (average votes, scaled)
n = len(y)
gradient_w = -(2/n) * (X_raw.T @ errors_init)
gradient_b = -(2/n) * errors_init.sum()

print("Mean gradient vector (direction to move each weight to REDUCE loss):")
print()
print(f"  {'Feature':<18} {'Mean vote':>14}  {'Gradient (−2/n × vote)':>24}  {'True weight':>12}")
print("  " + "-" * 74)
for feat, mv, gv, tw in zip(FEATURES, (X_raw.T @ errors_init)/n, gradient_w, TRUE_W):
    print(f"  {feat:<18} {mv:>14,.0f}  {gv:>24,.2f}  {tw:>+12,.0f}")
print()
print("All gradients are negative → subtract a negative → weights increase → correct direction.")


## Part 4 — The Update Rule and the Full Loop

After computing the gradient, we take a small step in the direction that **reduces** loss:

```
w_j  ←  w_j  −  learning_rate × gradient_j
```

The **learning rate** controls the step size:
- Too large: weights overshoot the minimum and the loss diverges
- Too small: training takes forever
- Just right: loss falls smoothly and plateaus at the minimum

Because features are on very different scales (age: 25–65, smoker: 0–1), we scale all features to
mean=0, std=1 before training. This ensures every weight gets updated at a similar rate.
After training we recover the raw-space weights using:

```
raw_weight_j   =  scaled_weight_j  /  std_j
raw_intercept  =  scaled_intercept − Σ( raw_weight_j × mean_j )
```

The full loop:

```
initialize: w = zeros,  b = 0

for each epoch:
    ŷ         = X_scaled @ w + b
    errors    = y − ŷ
    grad_w    = −(2/n) × X_scaled.T @ errors
    grad_b    = −(2/n) × sum(errors)
    w        −= learning_rate × grad_w
    b        −= learning_rate × grad_b
    record MSE
```


In [ ]:
scaler   = StandardScaler()
X_sc     = scaler.fit_transform(X_raw)

LR     = 0.05
EPOCHS = 2000

w = np.zeros(len(FEATURES))
b = 0.0

history_loss = []
history_w    = []
history_b    = []

for epoch in range(EPOCHS):
    y_pred  = X_sc @ w + b
    errors  = y - y_pred
    mse     = np.mean(errors ** 2)

    grad_w  = -(2 / n) * (X_sc.T @ errors)
    grad_b  = -(2 / n) * errors.sum()

    w -= LR * grad_w
    b -= LR * grad_b

    history_loss.append(mse)
    history_w.append(w.copy())
    history_b.append(b)

history_w = np.array(history_w)

# Recover raw-space weights
raw_weights        = w / scaler.scale_
raw_intercept      = b - np.dot(raw_weights, scaler.mean_)

print(f"Training complete: {EPOCHS} epochs   learning rate = {LR}")
print(f"Final MSE: {history_loss[-1]:,.0f}   RMSE: ${np.sqrt(history_loss[-1]):,.0f}")
print()
print(f"  {'Feature':<18} {'True weight':>12}  {'Learned weight':>14}  {'Difference':>12}")
print("  " + "-" * 62)
for feat, tw, lw in zip(FEATURES, TRUE_W, raw_weights):
    diff = lw - tw
    print(f"  {feat:<18} {tw:>+12,.1f}  {lw:>+14,.1f}  {diff:>+12,.1f}")
print(f"  {'intercept':<18} {TRUE_INTERCEPT:>+12,.1f}  {raw_intercept:>+14,.1f}  {raw_intercept-TRUE_INTERCEPT:>+12,.1f}")
print()
print("Learned weights are close to the true values — differences come from noise in the data.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Linear scale ─────────────────────────────────────────────────────────────
axes[0].plot(history_loss, color='#1565C0', lw=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE (loss)')
axes[0].set_title('Loss curve — linear scale\nFalls steeply at first, then flattens', fontsize=10)

# ── Log scale (shows early and late behaviour together) ───────────────────────
axes[1].semilogy(history_loss, color='#1565C0', lw=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE (log scale)')
axes[1].set_title('Loss curve — log scale\nStraight descent = exponential convergence', fontsize=10)

# Annotate 50% and 90% reduction milestones
initial_loss = history_loss[0]
for pct, color in [(0.50, '#FF7043'), (0.10, '#4CAF50')]:
    target = initial_loss * pct
    try:
        ep = next(i for i, l in enumerate(history_loss) if l <= target)
        for ax in axes:
            ax.axvline(ep, color=color, lw=1.2, linestyle='--',
                       label=f'{int((1-pct)*100)}% loss reduction @ epoch {ep}')
    except StopIteration:
        pass

axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.suptitle('MSE falls as gradient descent moves weights toward their true values',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f"Initial MSE:  {history_loss[0]:>12,.0f}")
print(f"After 100ep:  {history_loss[99]:>12,.0f}  ({100*(1-history_loss[99]/history_loss[0]):.0f}% reduction)")
print(f"After 500ep:  {history_loss[499]:>12,.0f}  ({100*(1-history_loss[499]/history_loss[0]):.0f}% reduction)")
print(f"Final MSE:    {history_loss[-1]:>12,.0f}  ({100*(1-history_loss[-1]/history_loss[0]):.0f}% reduction)")


In [ ]:
# Recover raw-space weight history at every epoch
raw_w_history = history_w / scaler.scale_   # shape (EPOCHS, n_features)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat = list(axes.flat)   # convert to list so it can be indexed after the loop

for ax, feat, tw, raw_w_trace in zip(axes_flat, FEATURES, TRUE_W, raw_w_history.T):
    epochs = np.arange(EPOCHS)
    ax.plot(epochs, raw_w_trace, color='#1565C0', lw=1.5, label='learned weight')
    ax.axhline(tw, color='#e57373', lw=1.8, linestyle='--', label=f'true = {tw:+,.0f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Weight value')
    ax.set_title(feat, fontsize=11)
    ax.legend(fontsize=8)

# Use last subplot for intercept
ax = axes_flat[5]
raw_int_history = np.array(history_b) - np.array([
    np.dot(history_w[e] / scaler.scale_, scaler.mean_) for e in range(EPOCHS)
])
ax.plot(np.arange(EPOCHS), raw_int_history, color='#7E57C2', lw=1.5, label='learned intercept')
ax.axhline(TRUE_INTERCEPT, color='#e57373', lw=1.8, linestyle='--',
           label=f'true = {TRUE_INTERCEPT:+,.0f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Intercept value')
ax.set_title('intercept', fontsize=11)
ax.legend(fontsize=8)

plt.suptitle('Weight trajectories — each weight converges toward its true value over training',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  w_smoker and w_has_chronic (binary features) converge more slowly than w_age and w_bmi.")
print("  Binary features are 0 for many rows — those rows contribute zero gradient for that weight.")
print("  The model can only update w_smoker from the 25% of rows where smoker=1.")


In [ ]:
# How many rows actually vote on each weight per epoch?
voting_rows = {
    'age':            (X_raw[:, 0] != 0).sum(),
    'bmi':            (X_raw[:, 1] != 0).sum(),
    'activity_level': (X_raw[:, 2] != 0).sum(),
    'smoker':         (X_raw[:, 3] != 0).sum(),
    'has_chronic':    (X_raw[:, 4] != 0).sum(),
}

print("How many observations vote on each weight per training step?")
print()
print(f"  {'Feature':<18} {'Type':<12} {'Voting rows':>12}  {'Abstaining':>12}  {'% active':>10}")
print("  " + "-" * 66)
for feat, ftype, vr in zip(FEATURES, FTYPE, voting_rows.values()):
    abst = n - vr
    pct  = 100 * vr / n
    print(f"  {feat:<18} {ftype:<12} {vr:>12}  {abst:>12}  {pct:>9.0f}%")

print()
print("Continuous and ordinal features: every row votes (value is almost never exactly 0).")
print("Binary features: only rows where the feature=1 vote — the rest abstain.")

# ── Show how binary sparsity slows convergence ────────────────────────────────
# Train two separate 1-feature models and compare convergence speed
def train_single_feature(feat_idx, epochs=2000, lr=0.05):
    X_f  = X_sc[:, feat_idx:feat_idx+1]
    w    = np.zeros(1)
    b    = 0.0
    loss = []
    for _ in range(epochs):
        yp = X_f @ w + b
        e  = y - yp
        w -= lr * (-(2/n) * (X_f.T @ e))
        b -= lr * (-(2/n) * e.sum())
        loss.append(np.mean(e**2))
    return loss

loss_age     = train_single_feature(0)  # age — always votes
loss_smoker  = train_single_feature(3)  # smoker — only 25% vote

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(loss_age,    color='#1565C0', lw=2, label='age  (continuous — all rows vote)')
ax.semilogy(loss_smoker, color='#FF7043', lw=2, label='smoker  (binary — only 25% of rows vote)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Single-feature convergence: age vs smoker\nBinary feature converges more slowly — fewer rows contribute gradient signal', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("The smoker-only model reaches a higher final MSE because one feature can't fully explain")
print("the variance — but notice it also converges more slowly per epoch. Sparse features need")
print("more data or more epochs to push their weights to the right value.")


In [ ]:
def run_gd(lr, epochs=500):
    w, b = np.zeros(len(FEATURES)), 0.0
    loss_hist = []
    for _ in range(epochs):
        yp = X_sc @ w + b
        e  = y - yp
        mse = np.mean(e**2)
        if mse > 1e15:          # diverged
            loss_hist += [float('nan')] * (epochs - len(loss_hist))
            break
        loss_hist.append(mse)
        w -= lr * (-(2/n) * (X_sc.T @ e))
        b -= lr * (-(2/n) * e.sum())
    return loss_hist

configs = [
    (0.001, '#9E9E9E', 'LR=0.001  (too small — barely moves)'),
    (0.05,  '#1565C0', 'LR=0.05   (just right — smooth convergence)'),
    (0.50,  '#e57373', 'LR=0.50   (too large — overshoots and diverges)'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for lr, color, label in configs:
    hist = run_gd(lr)
    axes[0].plot(hist,                       color=color, lw=1.8, label=label)
    axes[1].semilogy(hist,                   color=color, lw=1.8, label=label)

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)

axes[0].set_title('Learning rate comparison — linear scale', fontsize=10)
axes[1].set_title('Learning rate comparison — log scale', fontsize=10)
plt.suptitle('Too small: slow.   Just right: smooth.   Too large: diverges.',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print("Too small (0.001):  each update moves the weights by a tiny fraction — needs many more epochs")
print("Just right (0.05):  loss falls smoothly and plateaus at the minimum")
print("Too large (0.50):   each update overshoots — the new weights are worse than the old ones")
print("                    loss explodes to infinity (divergence)")


## Key Takeaways

### The four-step loop

```
initialize:  w = 0  for all features

repeat until convergence:
  1. predict    ŷᵢ  =  intercept  +  Σⱼ( wⱼ × xᵢⱼ )
  2. error      eᵢ  =  yᵢ − ŷᵢ
  3. gradient   ∇wⱼ =  −(2/n) × Σᵢ( eᵢ × xᵢⱼ )
  4. update     wⱼ  ←  wⱼ  −  lr × ∇wⱼ
```

### The silent feature rule — same in prediction AND learning

| Situation | Prediction | Gradient vote |
|---|---|---|
| feature value = 0 | contribution = 0 (silent) | vote = 0 (abstains) |
| feature value ≠ 0 | contribution = w × value | vote = error × value |

A feature that is silent in the prediction is also silent in the gradient.
Binary and categorical features are 0 for many rows — those rows cannot move that weight.

### Scale matters

- Without StandardScaler: large-valued features (age ~40, bmi ~29) get gradient votes 40–50× larger than binary features (0 or 1) — purely due to magnitude, not importance
- With StandardScaler: all features compete equally in the gradient update

### Learning rate

- Too small → many epochs needed, slow convergence
- Too large → weights overshoot, loss diverges
- Typical starting range: 0.01 – 0.1 for gradient descent on standardised features

### The connecting thread across all six notebooks

```
Notebooks 01–03:  understand your data before training
Notebook 04:      prediction = Σ( weight × value )
Notebook 05:      apply all four steps to one real dataset
Notebook 06:      learning = correcting weights using error × value
```

The arithmetic that makes a prediction and the arithmetic that improves weights are the same multiplication.
